# TweetSense Sentiment Comparison

In this notebook we compare two methods:

1. **Traditional Machine Learning:** clean text, TF-IDF, then train ML classifiers.
2. **Fine-tuned Twitter RoBERTa:** fine-tune `cardiffnlp/twitter-roberta-base` for our four sentiment labels.

Labels: `Positive`, `Negative`, `Neutral`, `Irrelevant`.

## 1. Import Libraries

In [ ]:
import re
import string
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

sns.set_theme(style="whitegrid")

## 2. Basic Settings

In [ ]:
TRAIN_PATH = r"C:\Users\nwf15\OneDrive\Desktop\NLP_Project\twitter_training.csv"
VALIDATION_PATH = r"C:\Users\nwf15\OneDrive\Desktop\NLP_Project\twitter_validation.csv"

LABELS = ["Positive", "Negative", "Neutral", "Irrelevant"]
RANDOM_STATE = 42

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

print("Training file exists:", Path(TRAIN_PATH).exists())
print("Validation file exists:", Path(VALIDATION_PATH).exists())

## 3. Download NLTK Resources

We use NLTK for stopwords, stemming, and lemmatization.

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

stop_words = set(stopwords.words("english"))
tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

## 4. Load and Split the Dataset

The dataset has four columns, so we give them clear names. We combine the available CSV files, then create a stratified split with 5,000 validation rows, 5,000 test rows, and the remaining rows for training.

In [ ]:
columns = ["tweet_id", "entity", "sentiment", "tweet"]

raw_train_df = pd.read_csv(TRAIN_PATH, header=None, names=columns)
raw_validation_df = pd.read_csv(VALIDATION_PATH, header=None, names=columns)
full_df = pd.concat([raw_train_df, raw_validation_df], ignore_index=True)

print("Original training file shape:", raw_train_df.shape)
print("Original validation file shape:", raw_validation_df.shape)
print("Combined dataset shape:", full_df.shape)
full_df.head()

## 5. Clean Missing Values, Labels, and Split

In [ ]:
VALIDATION_SIZE = 5000
TEST_SIZE = 5000


def keep_valid_rows(df):
    df = df.dropna(subset=["tweet", "sentiment"])
    df = df[df["sentiment"].isin(LABELS)]
    return df.reset_index(drop=True)


def stratified_split(df, test_size, validation_size):
    train_valid_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["sentiment"],
        random_state=RANDOM_STATE,
    )

    train_df, valid_df = train_test_split(
        train_valid_df,
        test_size=validation_size,
        stratify=train_valid_df["sentiment"],
        random_state=RANDOM_STATE,
    )

    return [split.reset_index(drop=True) for split in (train_df, valid_df, test_df)]


def label_distribution(df, split_name):
    counts = df["sentiment"].value_counts().reindex(LABELS, fill_value=0)
    percentages = df["sentiment"].value_counts(normalize=True).reindex(LABELS, fill_value=0) * 100

    return pd.DataFrame({
        "split": split_name,
        "label": LABELS,
        "count": counts.values,
        "percentage": percentages.round(2).values,
    })


def show_split_summary(splits):
    for split_name, df in splits.items():
        print(f"{split_name.title()} shape:", df.shape)
        display(df["sentiment"].value_counts())

    distribution_df = pd.concat(
        [label_distribution(df, split_name) for split_name, df in splits.items()],
        ignore_index=True,
    )

    print("Split label distribution check")
    display(distribution_df.pivot(index="label", columns="split", values="percentage"))


full_df = keep_valid_rows(full_df)
train_df, valid_df, test_df = stratified_split(full_df, TEST_SIZE, VALIDATION_SIZE)

splits = {
    "train": train_df,
    "validation": valid_df,
    "test": test_df,
}
show_split_summary(splits)

light_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_distribution_df = pd.concat(
    [
        label_distribution(train_df.iloc[fold_index], f"fold {fold_number}")
        for fold_number, (_, fold_index) in enumerate(light_cv.split(train_df, train_df["sentiment"]), start=1)
    ],
    ignore_index=True,
)

print("Light 3-fold CV distribution check on training data")
display(cv_distribution_df.pivot(index="label", columns="split", values="percentage"))


## 6. Text Preprocessing Function

This follows our pipeline:

Raw tweet -> remove noise -> lowercase -> tokenize -> remove stopwords -> stemming or lemmatization.

In [ ]:
def clean_tweet(text, method="lemma"):
    text = str(text)

    # Remove URLs and mentions
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    # Remove only the hashtag symbol, but keep the word
    text = re.sub(r"#(\w+)", r"\1", text)

    # Remove emojis and non-English symbols
    text = text.encode("ascii", "ignore").decode("ascii")

    # Lowercase and remove punctuation
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenize and remove stopwords
    tokens = tokenizer.tokenize(text)
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]

    # Normalize words
    if method == "stem":
        tokens = [stemmer.stem(word) for word in tokens]
    else:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)


example = "I LOVE this update!!! 😍 https://example.com #Amazing @user"
print("Stemmed:", clean_tweet(example, method="stem"))
print("Lemmatized:", clean_tweet(example, method="lemma"))

## 7. Create Clean Text Columns

In [ ]:
def add_clean_text_columns(df):
    df = df.copy()
    df["tweet_stem"] = df["tweet"].apply(lambda text: clean_tweet(text, method="stem"))
    df["tweet_lemma"] = df["tweet"].apply(lambda text: clean_tweet(text, method="lemma"))
    return df


train_df = add_clean_text_columns(train_df)
valid_df = add_clean_text_columns(valid_df)
test_df = add_clean_text_columns(test_df)

train_df[["tweet", "tweet_stem", "tweet_lemma"]].head()


## 8. Train Traditional ML Models

We test three classifiers with both stemmed and lemmatized tweets.

What was improved here:

- The repeated preprocessing code was simplified with loops and helper functions.
- Model selection now uses 5-fold stratified cross-validation instead of choosing only from one validation result.
- Validation is used for model selection, and the separate 5,000-row test set is used for final reporting.
- TF-IDF was made less likely to overfit by using `min_df=5`, `max_df=0.85`, `max_features=30000`, and `sublinear_tf=True`.
- Logistic Regression and Linear SVM use stronger regularization with `C=0.5`.
- The table includes an `overfit_gap`, which is `train_macro_f1 - validation_macro_f1`.

In [ ]:
TFIDF_PARAMS = {
    "ngram_range": (1, 2),
    "min_df": 3,
    "max_df": 0.95,
    "max_features": 30000,
    "sublinear_tf": True,
}

CLASSIFIERS = {
    "Logistic Regression": LogisticRegression(C=0.5, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(C=2.0, class_weight="balanced", random_state=RANDOM_STATE),
    "Naive Bayes": MultinomialNB(alpha=1.0),
}

TEXT_COLUMNS = {
    "stem": "tweet_stem",
    "lemma": "tweet_lemma",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "f1_macro", "f1_weighted"]


def build_model(classifier):
    return Pipeline([
        ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
        ("classifier", clone(classifier)),
    ])


def score_model(model, X_train, y_train, X_valid, y_valid):
    start_time = time.time()
    cv_scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    valid_predictions = model.predict(X_valid)
    train_macro_f1 = f1_score(y_train, train_predictions, labels=LABELS, average="macro")
    valid_macro_f1 = f1_score(y_valid, valid_predictions, labels=LABELS, average="macro")

    return {
        "cv_accuracy": cv_scores["test_accuracy"].mean(),
        "cv_macro_f1": cv_scores["test_f1_macro"].mean(),
        "cv_weighted_f1": cv_scores["test_f1_weighted"].mean(),
        "train_macro_f1": train_macro_f1,
        "validation_accuracy": accuracy_score(y_valid, valid_predictions),
        "validation_macro_f1": valid_macro_f1,
        "validation_weighted_f1": f1_score(y_valid, valid_predictions, labels=LABELS, average="weighted"),
        "overfit_gap": train_macro_f1 - valid_macro_f1,
        "runtime_seconds": time.time() - start_time,
    }


def train_traditional_models():
    results = []
    models = {}
    y_train = train_df["sentiment"]
    y_valid = valid_df["sentiment"]

    for text_method, text_column in TEXT_COLUMNS.items():
        for classifier_name, classifier in CLASSIFIERS.items():
            model_name = f"{text_method.title()} + {classifier_name}"
            print("Training:", model_name)

            model = build_model(classifier)
            scores = score_model(
                model,
                train_df[text_column],
                y_train,
                valid_df[text_column],
                y_valid,
            )

            results.append({
                "method": "Traditional TF-IDF",
                "model": model_name,
                "text_method": text_method,
                **scores,
            })
            models[model_name] = model

    results_df = pd.DataFrame(results).sort_values("cv_macro_f1", ascending=False)
    return results_df, models


y_train = train_df["sentiment"]
y_valid = valid_df["sentiment"]
y_test = test_df["sentiment"]

traditional_results_df, saved_models = train_traditional_models()
traditional_results_df


## 9. Evaluate and Save the Best Traditional Model

In [ ]:
best_row = traditional_results_df.iloc[0]
best_model_name = best_row["model"]
best_text_method = best_row["text_method"]
best_model = saved_models[best_model_name]

test_clean_text = test_df[f"tweet_{best_text_method}"]
traditional_predictions = best_model.predict(test_clean_text)

print("Best traditional model:", best_model_name)
print(f"CV macro F1: {best_row['cv_macro_f1']:.3f}")
print(f"Validation macro F1: {best_row['validation_macro_f1']:.3f}")
print(f"Test macro F1: {f1_score(y_test, traditional_predictions, labels=LABELS, average='macro'):.3f}")
print(f"Overfit gap: {best_row['overfit_gap']:.3f}")
print(classification_report(y_test, traditional_predictions, labels=LABELS))

model_package = {
    "model": best_model,
    "text_method": best_text_method,
    "labels": LABELS,
}

model_path = MODEL_DIR / "tweetsense_tfidf_model.joblib"
joblib.dump(model_package, model_path)
print("Saved model package to:", model_path)

In [ ]:
def show_confusion_matrix(y_true, y_pred, title):
    matrix = confusion_matrix(y_true, y_pred, labels=LABELS)

    plt.figure(figsize=(7, 5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


show_confusion_matrix(y_test, traditional_predictions, best_model_name)

## 10. Fine-Tune Twitter RoBERTa

Instead of using a ready-made Hugging Face classification pipeline, we fine-tune the base `cardiffnlp/twitter-roberta-base` model on this dataset.

The training loop tests two practical hyperparameter sets using 12,000 and 20,000 stratified training examples, evaluates on 3,000 validation examples, and keeps the checkpoint with the best validation macro F1 score.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, set_seed

HF_BASE_MODEL_NAME = "cardiffnlp/twitter-roberta-base"
HF_OUTPUT_DIR = MODEL_DIR / "tweetsense_twitter_roberta"
HF_MAX_LENGTH = 128

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}
set_seed(RANDOM_STATE)

tokenizer_roberta = AutoTokenizer.from_pretrained(HF_BASE_MODEL_NAME)


def build_roberta_text(df):
    entity = df["entity"].astype(str)
    tweet = df["tweet"].astype(str)
    return ("Tweet about " + entity + ": " + tweet).tolist()


class TweetSentimentDataset(Dataset):
    def __init__(self, df):
        self.encodings = tokenizer_roberta(
            build_roberta_text(df),
            truncation=True,
            padding=True,
            max_length=HF_MAX_LENGTH,
        )
        self.labels = [label2id[label] for label in df["sentiment"]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index])
        return item


ROBERTA_VALIDATION_SIZE = 3000


def stratified_sample(df, sample_size, random_state):
    if sample_size >= len(df):
        return df.reset_index(drop=True)

    sampled_df, _ = train_test_split(
        df,
        train_size=sample_size,
        stratify=df["sentiment"],
        random_state=random_state,
    )
    return sampled_df.reset_index(drop=True)


roberta_valid_df = stratified_sample(valid_df, ROBERTA_VALIDATION_SIZE, RANDOM_STATE)
roberta_valid_dataset = TweetSentimentDataset(roberta_valid_df)
roberta_test_dataset = TweetSentimentDataset(test_df)

print("Validation examples:", len(roberta_valid_dataset))
print("Test examples:", len(roberta_test_dataset))


In [ ]:
import inspect


def compute_roberta_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }


ROBERTA_RUNS = [
    {"train_size": 12000, "learning_rate": 3e-5, "per_device_train_batch_size": 32, "num_train_epochs": 1, "weight_decay": 0.01, "warmup_ratio": 0.06},
    {"train_size": 20000, "learning_rate": 2e-5, "per_device_train_batch_size": 32, "num_train_epochs": 1, "weight_decay": 0.01, "warmup_ratio": 0.06},
]


def create_roberta_model():
    return AutoModelForSequenceClassification.from_pretrained(
        HF_BASE_MODEL_NAME,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
    )


def training_argument_names():
    return set(inspect.signature(TrainingArguments.__init__).parameters)


def add_if_supported(args, name, value, supported_names):
    if name in supported_names:
        args[name] = value


def create_training_args(run_number, params):
    supported_names = training_argument_names()
    args = {
        "output_dir": str(HF_OUTPUT_DIR / f"run_{run_number}"),
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "logging_steps": 100,
        "save_total_limit": 1,
        "report_to": "none",
        "seed": RANDOM_STATE,
        "fp16": torch.cuda.is_available(),
    }

    evaluation_arg = "eval_strategy" if "eval_strategy" in supported_names else "evaluation_strategy"
    args[evaluation_arg] = "epoch"

    add_if_supported(args, "overwrite_output_dir", True, supported_names)
    add_if_supported(args, "optim", "adamw_torch", supported_names)

    trainer_params = {key: value for key, value in params.items() if key != "train_size"}

    for name, value in trainer_params.items():
        add_if_supported(args, name, value, supported_names)

    return TrainingArguments(**args)


def fine_tune_roberta(run_number, params):
    print(f"Fine-tuning run {run_number}:", params)
    roberta_train_df = stratified_sample(train_df, params["train_size"], RANDOM_STATE + run_number)
    roberta_train_dataset = TweetSentimentDataset(roberta_train_df)
    print("Training examples:", len(roberta_train_dataset))
    print("Validation examples:", len(roberta_valid_dataset))

    trainer = Trainer(
        model=create_roberta_model(),
        args=create_training_args(run_number, params),
        train_dataset=roberta_train_dataset,
        eval_dataset=roberta_valid_dataset,
        compute_metrics=compute_roberta_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    return {
        "run": run_number,
        **params,
        "validation_accuracy": metrics["eval_accuracy"],
        "validation_macro_f1": metrics["eval_macro_f1"],
        "validation_weighted_f1": metrics["eval_weighted_f1"],
        "trainer": trainer,
    }


roberta_tuning_results = []
best_roberta_result = None
start_time = time.time()

for run_number, params in enumerate(ROBERTA_RUNS, start=1):
    result = fine_tune_roberta(run_number, params)
    roberta_tuning_results.append({key: value for key, value in result.items() if key != "trainer"})

    if best_roberta_result is None or result["validation_macro_f1"] > best_roberta_result["validation_macro_f1"]:
        best_roberta_result = result

roberta_runtime = time.time() - start_time
roberta_tuning_results_df = pd.DataFrame(roberta_tuning_results).sort_values("validation_macro_f1", ascending=False)
roberta_tuning_results_df


## 11. Evaluate Fine-Tuned Twitter RoBERTa

In [ ]:
best_roberta_trainer = best_roberta_result["trainer"]
roberta_output = best_roberta_trainer.predict(roberta_test_dataset)
roberta_prediction_ids = np.argmax(roberta_output.predictions, axis=-1)
roberta_predictions = [id2label[prediction_id] for prediction_id in roberta_prediction_ids]

print("Best Twitter RoBERTa hyperparameters:")
display(pd.DataFrame([{key: value for key, value in best_roberta_result.items() if key != "trainer"}]))

print(classification_report(test_df["sentiment"], roberta_predictions, labels=LABELS))
show_confusion_matrix(test_df["sentiment"], roberta_predictions, "Fine-tuned cardiffnlp/twitter-roberta-base")

roberta_model_path = HF_OUTPUT_DIR / "best_model"
best_roberta_trainer.save_model(str(roberta_model_path))
tokenizer_roberta.save_pretrained(str(roberta_model_path))
print("Saved fine-tuned Twitter RoBERTa model to:", roberta_model_path)

roberta_metrics = {
    "method": "Fine-tuned Twitter RoBERTa",
    "model": HF_BASE_MODEL_NAME,
    "accuracy": accuracy_score(test_df["sentiment"], roberta_predictions),
    "macro_f1": f1_score(test_df["sentiment"], roberta_predictions, labels=LABELS, average="macro"),
    "weighted_f1": f1_score(test_df["sentiment"], roberta_predictions, labels=LABELS, average="weighted"),
    "runtime_seconds": roberta_runtime,
}

## 12. Final Comparison Table

In [ ]:
traditional_best_metrics = {
    "method": best_row["method"],
    "model": best_row["model"],
    "accuracy": accuracy_score(y_test, traditional_predictions),
    "macro_f1": f1_score(y_test, traditional_predictions, labels=LABELS, average="macro"),
    "weighted_f1": f1_score(y_test, traditional_predictions, labels=LABELS, average="weighted"),
    "runtime_seconds": best_row["runtime_seconds"],
    "notes": f"Selected by 5-fold CV macro F1={best_row['cv_macro_f1']:.3f}; final metrics are on the 5,000-row test set.",
}

roberta_metrics["notes"] = f"Fine-tuned from {HF_BASE_MODEL_NAME}; selected run {best_roberta_result['run']} by validation macro F1; final metrics are on the 5,000-row test set."

comparison_df = pd.DataFrame([traditional_best_metrics, roberta_metrics])
comparison_df[["method", "model", "accuracy", "macro_f1", "weighted_f1", "runtime_seconds", "notes"]]

## 13. Try the Saved Traditional Model

In [ ]:
loaded_package = joblib.load(model_path)
loaded_model = loaded_package["model"]
loaded_text_method = loaded_package["text_method"]

new_tweets = [
    "I love this game. The new update is amazing!",
    "This service is terrible and keeps crashing.",
    "The product page was updated today.",
    "I made lunch while everyone was talking about the match.",
]

clean_new_tweets = [clean_tweet(tweet, method=loaded_text_method) for tweet in new_tweets]
new_predictions = loaded_model.predict(clean_new_tweets)

pd.DataFrame({"tweet": new_tweets, "prediction": new_predictions})